In [ ]:
import polars as pl

import nwec.utility_reporting.arrearage_counts
import nwec.utility_reporting.arrearages
import nwec.utils.excel
from nwec.constants import RAW_UTILITY_DATA, Utility

YEAR = 2024
QUARTER = 4
NUM_MONTHS = 12
COLS_PER_MONTH = 1
SHEET_SEARCH_STRING = "past due balances"
ARREARAGE_SEARCH_STRING = "number of customers"
spreadsheet = RAW_UTILITY_DATA / str(YEAR) / f"{Utility.AVISTA.code}_{YEAR}_Q{QUARTER}.xlsx"
source_date_format = "%Y-%m-%d %H:%M:%S"

In [2]:
sheet_index = nwec.utils.excel.get_sheet_index_from_name(spreadsheet, SHEET_SEARCH_STRING)
df = pl.read_excel(spreadsheet, sheet_id=sheet_index, has_header=False)
arrearage_counts = nwec.utility_reporting.arrearages.get_arrearages_df(
    df, NUM_MONTHS, COLS_PER_MONTH, ARREARAGE_SEARCH_STRING
)


# Arrearage Counts


In [3]:
date_row = nwec.utility_reporting.arrearages.infer_date_row(arrearage_counts, source_date_format)
arrearage_counts = arrearage_counts.tail(-date_row)  # remove rows before the date row
arrearage_counts = nwec.utility_reporting.arrearage_counts.format_arrearage_count_dates(
    arrearage_counts, source_date_format
)
arrearage_counts = nwec.utility_reporting.arrearages.add_zip_and_customer_class_cols(df, arrearage_counts)
arrearage_counts = nwec.utility_reporting.arrearage_counts.normalize_arrearage_count_cols(
    arrearage_counts, Utility.AVISTA
)


# Save Results


In [4]:
nwec.utility_reporting.arrearage_counts.save_processed_arrearage_counts(arrearage_counts)